# Grant Keyword Search

Fetch posted opportunities from grants.gov for multiple keywords, count keyword matches per grant, and export the aggregated results to Excel.

In [5]:
import requests
from collections import Counter, defaultdict
from datetime import datetime
from html import unescape
from pathlib import Path
from xml.sax.saxutils import escape
from zipfile import ZipFile, ZIP_DEFLATED


In [6]:
KEYWORDS = [
    'HERS',
    'energy',
    'efficiency',
    'ratings',
    'decarbonization',
    'building',
    'certifications',
    'Massachusetts',
    'rebates',
    'consulting'
]
ROWS_PER_KEYWORD = 50
OUTPUT_PATH = Path('IvanLuciaWork/grant_keyword_matches2.xlsx')


In [7]:
def col_letter(idx: int) -> str:
    idx += 1
    letters = []
    while idx:
        idx, rem = divmod(idx - 1, 26)
        letters.append(chr(65 + rem))
    return ''.join(reversed(letters))


def build_cell(value, row_idx: int, col_idx: int) -> str:
    if value is None or value == '':
        return ''
    ref = f"{col_letter(col_idx)}{row_idx}"
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return f'<c r="{ref}"><v>{value}</v></c>'
    text = escape(str(value))
    return f'<c r="{ref}" t="inlineStr"><is><t>{text}</t></is></c>'


def build_sheet_xml(columns, rows):
    total_rows = len(rows) + 1 if rows else 1
    last_col_letter = col_letter(len(columns) - 1) if columns else 'A'
    dimension = f"A1:{last_col_letter}{total_rows}"

    header_cells = ''.join(build_cell(col, 1, idx) for idx, col in enumerate(columns))
    data_rows = []
    for r_index, row in enumerate(rows, start=2):
        cells = []
        for c_index, col in enumerate(columns):
            cell_xml = build_cell(row.get(col, ''), r_index, c_index)
            if cell_xml:
                cells.append(cell_xml)
        data_rows.append(f"<row r='{r_index}'>{''.join(cells)}</row>")

    sheet_xml = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">',
        f'<dimension ref="{dimension}"/>',
        '<sheetViews><sheetView workbookViewId="0"/></sheetViews>',
        '<sheetFormatPr defaultRowHeight="15"/>',
        '<sheetData>',
        f'<row r="1">{header_cells}</row>',
        *data_rows,
        '</sheetData>',
        '</worksheet>'
    ])
    return sheet_xml


def write_excel(path: Path, columns, rows):
    timestamp = datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
    sheet_xml = build_sheet_xml(columns, rows)

    content_types = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">',
        '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>',
        '<Default Extension="xml" ContentType="application/xml"/>',
        '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>',
        '<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>',
        '<Override PartName="/xl/styles.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.styles+xml"/>',
        '<Override PartName="/docProps/core.xml" ContentType="application/vnd.openxmlformats-package.core-properties+xml"/>',
        '<Override PartName="/docProps/app.xml" ContentType="application/vnd.openxmlformats-officedocument.extended-properties+xml"/>',
        '</Types>'
    ])

    rels = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">',
        '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>',
        '<Relationship Id="rId2" Type="http://schemas.openxmlformats.org/package/2006/relationships/metadata/core-properties" Target="docProps/core.xml"/>',
        '<Relationship Id="rId3" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/extended-properties" Target="docProps/app.xml"/>',
        '</Relationships>'
    ])

    workbook_rels = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">',
        '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>',
        '<Relationship Id="rId2" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/styles" Target="styles.xml"/>',
        '</Relationships>'
    ])

    workbook = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">',
        '<fileVersion appName="xl"/>',
        '<sheets>',
        '<sheet name="GrantMatches" sheetId="1" r:id="rId1"/>',
        '</sheets>',
        '</workbook>'
    ])

    styles = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<styleSheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">',
        '<fonts count="1"><font><sz val="11"/><color theme="1"/><name val="Calibri"/><family val="2"/></font></fonts>',
        '<fills count="1"><fill><patternFill patternType="none"/></fill></fills>',
        '<borders count="1"><border><left/><right/><top/><bottom/><diagonal/></border></borders>',
        '<cellStyleXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0"/></cellStyleXfs>',
        '<cellXfs count="1"><xf numFmtId="0" fontId="0" fillId="0" borderId="0" xfId="0"/></cellXfs>',
        '<cellStyles count="1"><cellStyle name="Normal" xfId="0" builtinId="0"/></cellStyles>',
        '</styleSheet>'
    ])

    core = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<cp:coreProperties xmlns:cp="http://schemas.openxmlformats.org/package/2006/metadata/core-properties" xmlns:dc="http://purl.org/dc/elements/1.1/" xmlns:dcterms="http://purl.org/dc/terms/" xmlns:dcmitype="http://purl.org/dc/dcmitype/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">',
        '<dc:title>Grant Keyword Matches</dc:title>',
        '<dc:creator>Codex Automation</dc:creator>',
        '<cp:lastModifiedBy>Codex Automation</cp:lastModifiedBy>',
        f'<dcterms:created xsi:type="dcterms:W3CDTF">{timestamp}</dcterms:created>',
        f'<dcterms:modified xsi:type="dcterms:W3CDTF">{timestamp}</dcterms:modified>',
        '</cp:coreProperties>'
    ])

    app = '\n'.join([
        '<?xml version="1.0" encoding="UTF-8"?>',
        '<Properties xmlns="http://schemas.openxmlformats.org/officeDocument/2006/extended-properties" xmlns:vt="http://schemas.openxmlformats.org/officeDocument/2006/docPropsVTypes">',
        '<Application>Microsoft Excel</Application>',
        '</Properties>'
    ])

    path.parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(path, 'w', ZIP_DEFLATED) as zf:
        zf.writestr('[Content_Types].xml', content_types)
        zf.writestr('_rels/.rels', rels)
        zf.writestr('xl/_rels/workbook.xml.rels', workbook_rels)
        zf.writestr('xl/workbook.xml', workbook)
        zf.writestr('xl/styles.xml', styles)
        zf.writestr('xl/worksheets/sheet1.xml', sheet_xml)
        zf.writestr('docProps/core.xml', core)
        zf.writestr('docProps/app.xml', app)


In [8]:
match_counts = Counter()
keyword_hits = defaultdict(set)
grants = {}
keyword_totals = {}

for keyword in KEYWORDS:
    payload = {
        'rows': ROWS_PER_KEYWORD,
        'keyword': keyword,
        'oppNum': '',
        'eligibilities': '',
        'agencies': '',
        'oppStatuses': 'posted',
        'aln': '',
        'fundingCategories': ''
    }
    response = requests.post(
        'https://api.grants.gov/v1/api/search2',
        headers={'Content-Type': 'application/json'},
        json=payload,
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    hits = data.get('data', {}).get('oppHits', [])
    keyword_totals[keyword] = len(hits)
    for hit in hits:
        grant_id = hit.get('id')
        if not grant_id:
            continue
        match_counts[grant_id] += 1
        keyword_hits[grant_id].add(keyword)
        if grant_id not in grants:
            info = hit.copy()
            cfda = info.get('cfdaList', [])
            if isinstance(cfda, list):
                info['cfdaList'] = ', '.join(cfda)
            elif cfda is None:
                info['cfdaList'] = ''
            else:
                info['cfdaList'] = str(cfda)
            grants[grant_id] = info

columns = ['Match', 'MatchedKeywords', 'GrantID', 'OpportunityNumber', 'Title', 'AgencyCode',
           'Agency', 'OpenDate', 'CloseDate', 'Status', 'DocType', 'CFDA']
rows = []
for grant_id, info in grants.items():
    rows.append({
        'Match': match_counts[grant_id],
        'MatchedKeywords': ', '.join(sorted(keyword_hits[grant_id])),
        'GrantID': str(grant_id),
        'OpportunityNumber': unescape(info.get('number', '') or ''),
        'Title': unescape(info.get('title', '') or ''),
        'AgencyCode': unescape(info.get('agencyCode', '') or ''),
        'Agency': unescape(info.get('agency', '') or ''),
        'OpenDate': unescape(info.get('openDate', '') or ''),
        'CloseDate': unescape(info.get('closeDate', '') or ''),
        'Status': unescape(info.get('oppStatus', '') or ''),
        'DocType': unescape(info.get('docType', '') or ''),
        'CFDA': unescape(info.get('cfdaList', '') or '')
    })

rows.sort(key=lambda r: (-r['Match'], r['CloseDate'], r['Title']))

write_excel(OUTPUT_PATH, columns, rows)

print(f'Created Excel file with {len(rows)} grants at {OUTPUT_PATH}')
print('\nHits per keyword:')
for keyword in KEYWORDS:
    print(f'  {keyword}: {keyword_totals.get(keyword, 0)}')

print('\nTop 5 grants by keyword matches:')
for row in rows[:5]:
    print(
        f"  Match={row['Match']} | Title={row['Title']} | Keywords={row['MatchedKeywords']}"
    )


Created Excel file with 209 grants at IvanLuciaWork\grant_keyword_matches2.xlsx

Hits per keyword:
  HERS: 46
  energy: 50
  efficiency: 50
  ratings: 41
  decarbonization: 1
  building: 50
  certifications: 50
  Massachusetts: 10
  rebates: 10
  consulting: 40

Top 5 grants by keyword matches:
  Match=5 | Title=Boosting Innovative GEOINT - Science & Technology  Broad Agency Announcement (BIG-ST BAA) | Keywords=HERS, certifications, consulting, ratings, rebates
  Match=4 | Title=FY 2021 - 2023 Economic Development RNTA | Keywords=HERS, certifications, efficiency, ratings
  Match=4 | Title=Long Range Broad Agency Announcement (BAA) for NSWC Crane | Keywords=certifications, consulting, efficiency, energy
  Match=4 | Title=UNITED STATES MILITARY ACADEMY Broad Agency Announcement | Keywords=HERS, certifications, efficiency, energy
  Match=4 | Title=Research and Development (RAD) Directed Energy (RD) University Assistance Instruments | Keywords=HERS, consulting, energy, rebates
